# FrenchyShona Figure 9 — Grounding and disagreement audit

This notebook creates the journal figure from frozen audit summaries. It does **not** rerun any model, LLM, translation, candidate screen, or classifier.

Upload/extract `FrenchyShona_Grounding_Audit_Figure9_Data.zip` as a Kaggle dataset.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import zipfile

INPUT = Path('/kaggle/input/frenchyshona-grounding-audit-figure9-data')
OUTPUT = Path('/kaggle/working/FrenchyShona_Figure9')
OUTPUT.mkdir(parents=True, exist_ok=True)

ghr = pd.read_csv(INPUT / 'grounding_hallucination_rate_summary.csv')
mrdr = pd.read_csv(INPUT / 'model_reviewer_disagreement_summary.csv')
display(ghr)
display(mrdr)


In [ ]:
# Verify the locked values before drawing.
assert ghr['audited'].tolist() == [32, 5, 37]
assert ghr['ungrounded'].tolist() == [0, 0, 0]
assert ghr['ci_high_percent'].round(2).tolist() == [10.72, 43.45, 9.41]
assert int(mrdr.loc[0, 'comparable']) == 18
assert int(mrdr.loc[0, 'differing']) == 2
assert round(float(mrdr.loc[0, 'rate_percent']), 2) == 11.11
print('Ground-truth checks passed.')


In [ ]:
def make_figure(grayscale=False):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 2.7))

    # Panel (a): GHR. Reverse order so Combined appears at top.
    order = [2, 1, 0]
    labels = ['Combined\n(n=37)', 'Ciluba completion\n(n=5)', 'Shona recovery\n(n=32)']
    upper = ghr.loc[order, 'ci_high_percent'].to_numpy()
    y = range(3)
    if grayscale:
        ax1.errorbar([0,0,0], y, xerr=[ [0,0,0], upper ], fmt='o', capsize=3, color='0.30', ecolor='0.45')
    else:
        ax1.errorbar([0,0,0], y, xerr=[ [0,0,0], upper ], fmt='o', capsize=3)
    ax1.set_yticks(list(y), labels)
    ax1.invert_yaxis()
    ax1.set_xlim(-1.5, 50)
    ax1.set_xlabel('Grounding-based hallucination rate (%)')
    ax1.set_title('(a) Grounding audit')
    ax1.grid(axis='x', alpha=0.20)

    # Panel (b): pooled model-reviewer disagreement.
    rate = float(mrdr.loc[0, 'rate_percent'])
    lo = float(mrdr.loc[0, 'ci_low_percent'])
    hi = float(mrdr.loc[0, 'ci_high_percent'])
    if grayscale:
        ax2.errorbar(rate, 0, xerr=[[rate-lo], [hi-rate]], fmt='o', capsize=3, color='0.30', ecolor='0.45')
    else:
        ax2.errorbar(rate, 0, xerr=[[rate-lo], [hi-rate]], fmt='o', capsize=3)
    ax2.set_yticks([0], ['Core comparison\n(n=18)'])
    ax2.set_ylim(-0.8, 0.8)
    ax2.set_xlim(-1.5, 50)
    ax2.set_xlabel('Model-reviewer disagreement rate (%)')
    ax2.set_title('(b) Assessment disagreement')
    ax2.grid(axis='x', alpha=0.20)

    for ax in (ax1, ax2):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.tight_layout()
    return fig

outputs = []
for suffix, grayscale in [('colour', False), ('grayscale', True)]:
    fig = make_figure(grayscale=grayscale)
    pdf = OUTPUT / f'fig_5_11_grounding_audit_{suffix}.pdf'
    png = OUTPUT / f'fig_5_11_grounding_audit_{suffix}.png'
    fig.savefig(pdf, bbox_inches='tight')
    fig.savefig(png, dpi=300, bbox_inches='tight')
    plt.show()
    outputs.extend([pdf, png])

print('\n'.join(str(p) for p in outputs))


In [ ]:
# Package both figure versions into one file for easy Kaggle download.
zip_path = Path('/kaggle/working/FrenchyShona_Figure9_Grounding_Audit_Both_Versions.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in outputs:
        zf.write(p, arcname=p.name)
print('Download from Kaggle Output/Files:')
print(zip_path)
